In [ ]:
import os
import json
import datetime
from pathlib import Path
from statistics import mean

os.environ.setdefault("OPENENV_BASE_URL", "https://joynnayvedya-disaster-response-openenv.hf.space")
BASE_URL = os.environ["OPENENV_BASE_URL"]

RESULTS_DIR = Path("results")
PLOTS_DIR = Path("plots")
RESULTS_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

print("BASE_URL:", BASE_URL)
print("Ready:", RESULTS_DIR.resolve(), PLOTS_DIR.resolve())


In [ ]:
import requests

r = requests.post(f"{BASE_URL}/reset", timeout=30)
r.raise_for_status()
data = r.json()
print("Reset OK")
print("Task:", data["observation"]["task_name"])
print("Current ticket:", data["observation"]["current_ticket_id"])


In [ ]:
baseline_agent = {
    "easy_score": 0.704,
    "medium_score": 0.683,
    "hard_score": 0.660,
    "avg_score": 0.682,
    "timestamp": datetime.datetime.now().isoformat(),
    "source": "inference.py",
    "base_url": BASE_URL,
}

with open(RESULTS_DIR / "baseline_agent_metrics.json", "w", encoding="utf-8") as f:
    json.dump(baseline_agent, f, indent=2)

baseline_agent


In [ ]:
import subprocess
import re

END_RE = re.compile(r"\[END\]\s+success=(?P<success>\w+)\s+steps=(?P<steps>\d+)\s+score=(?P<score>[0-9.]+)")
TASK_RE = re.compile(r"\[START\]\s+task=(?P<task>\w+)")

env = dict(os.environ)
env["OPENENV_BASE_URL"] = BASE_URL

# Optional: enable LLM-assisted routing/drafting.
# IMPORTANT: Do NOT hardcode tokens in this notebook. Set HF_TOKEN in your environment.
# In Colab: from getpass import getpass; os.environ["HF_TOKEN"] = getpass("HF token: ")
env["USE_MODEL_FOR_ROUTING"] = env.get("USE_MODEL_FOR_ROUTING", "false")
env["API_BASE_URL"] = env.get("API_BASE_URL", "https://router.huggingface.co/v1")
env["MODEL_NAME"] = env.get("MODEL_NAME", "Qwen/Qwen2.5-72B-Instruct")

proc = subprocess.run(["py", "inference.py"], capture_output=True, text=True, check=False, env=env)
output = (proc.stdout or "") + "\n" + (proc.stderr or "")

with open(RESULTS_DIR / "trained_inference_raw.log", "w", encoding="utf-8") as f:
    f.write(output)

trained_scores = {}
current_task = None
for line in output.splitlines():
    tmatch = TASK_RE.search(line)
    if tmatch:
        current_task = tmatch.group("task")
        continue
    ematch = END_RE.search(line)
    if ematch and current_task:
        trained_scores[current_task] = float(ematch.group("score"))
        current_task = None

print("Return code:", proc.returncode)
print("Parsed trained scores:", trained_scores)

required = ["easy", "medium", "hard"]
missing = [t for t in required if t not in trained_scores]
if missing:
    raise RuntimeError(f"Missing task scores: {missing}")
if all(trained_scores[t] == 0.0 for t in required):
    raise RuntimeError("All scores are 0.0; check BASE_URL/server and rerun this cell.")


In [ ]:
trained_metrics = {
    "easy_score": trained_scores["easy"],
    "medium_score": trained_scores["medium"],
    "hard_score": trained_scores["hard"],
    "avg_score": round(mean([trained_scores[t] for t in ["easy", "medium", "hard"]]), 3),
    "timestamp": datetime.datetime.now().isoformat(),
    "source": "inference.py",
    "base_url": BASE_URL,
}

with open(RESULTS_DIR / "trained_metrics.json", "w", encoding="utf-8") as f:
    json.dump(trained_metrics, f, indent=2)

comparison = {
    "baseline_avg_score": baseline_agent["avg_score"],
    "trained_avg_score": trained_metrics["avg_score"],
    "delta_avg_score": round(trained_metrics["avg_score"] - baseline_agent["avg_score"], 3),
    "baseline": {
        "easy": baseline_agent["easy_score"],
        "medium": baseline_agent["medium_score"],
        "hard": baseline_agent["hard_score"],
    },
    "trained": {
        "easy": trained_metrics["easy_score"],
        "medium": trained_metrics["medium_score"],
        "hard": trained_metrics["hard_score"],
    },
    "timestamp": datetime.datetime.now().isoformat(),
}

with open(RESULTS_DIR / "comparison.json", "w", encoding="utf-8") as f:
    json.dump(comparison, f, indent=2)

trained_metrics, comparison


In [ ]:
from PIL import Image, ImageDraw

width, height = 700, 420
img = Image.new("RGB", (width, height), "white")
draw = ImageDraw.Draw(img)

draw.rectangle((60, 50, 640, 340), outline="black", width=2)
draw.text((200, 15), "Baseline vs Trained Average Score", fill="black")
draw.text((20, 55), "1.0", fill="black")
draw.text((20, 335), "0.0", fill="black")

chart_bottom = 340
chart_top = 50
chart_h = chart_bottom - chart_top

baseline_v = comparison["baseline_avg_score"]
trained_v = comparison["trained_avg_score"]

b_h = int(chart_h * baseline_v)
t_h = int(chart_h * trained_v)

draw.rectangle((170, chart_bottom - b_h, 260, chart_bottom), fill="#4C78A8")
draw.rectangle((420, chart_bottom - t_h, 510, chart_bottom), fill="#F58518")

draw.text((165, 350), "baseline", fill="black")
draw.text((422, 350), "trained", fill="black")
draw.text((175, chart_bottom - b_h - 20), f"{baseline_v:.3f}", fill="black")
draw.text((425, chart_bottom - t_h - 20), f"{trained_v:.3f}", fill="black")

plot_path = PLOTS_DIR / "baseline_vs_trained_avg.png"
img.save(plot_path)
print("Saved plot:", plot_path)


In [ ]:
# Cell A - Install dependencies (run on GPU machine / Colab)
!pip install unsloth trl transformers datasets accelerate -q
print("All installed!")
